### SRT to Speaker-Labeled TXT Conversion

This notebook converts `.srt` subtitle files from dyadic conversations into plain-text transcripts with one utterance per line and a clear speaker label (e.g., `PARTICIPANT: text`). The goal is to produce reproducible, analysis-ready dialogue data for research.

**Inputs**
- SRT files located under `SRT_DIR` (default: `recording/all_srt`).
- Filenames are assumed to contain dyad identifiers (e.g., `dyad2_...`) and condition labels (`cloudy`, `piper`).

**Outputs**
- For each `.srt` file, a corresponding `.txt` file written next to it, with lines of the form:
  - `SPEAKER: utterance`
- Optional metadata columns (dyad, condition, timing) are available in intermediate `pandas` DataFrames for further analysis.

**Key processing steps**
- Parse dyad ID and condition from each filename.
- Parse and clean caption text, extracting speaker labels from patterns like `<v Speaker>`, `[Speaker]:`, or `Speaker:`.
- Optionally merge very short same-speaker fragments within a turn (disabled by default to preserve turn-taking).
- Remove exact duplicate captions to avoid repeated lines.

**Reproducibility notes**
- Python packages: `pysrt`, `pandas`, `numpy` (see your environment or `requirements.txt` if present).
- The notebook is designed to run top-to-bottom:
  1. Set `SRT_DIR`.
  2. Run the cell that discovers all `.srt` files.
  3. Run the cell that calls `srt_to_labeled_txt` for one file or for all files.
- Randomness is not used; output is deterministic given the same inputs and code version.

In [1]:
import os, re
import pysrt
import pandas as pd
import numpy as np

SRT_DIR = "/Users/Meihui/Downloads/sync/CVS/recording/all_srt"   # your folder

# -----------------------------
# 1) Parse metadata from path
# -----------------------------
def parse_filename_from_path(fp: str):
    base = os.path.basename(fp)
    stem = re.sub(r"\.srt$", "", base, flags=re.IGNORECASE)
    m = re.search(r"dyad(\d+)", stem, flags=re.IGNORECASE)
    dyad_id = int(m.group(1)) if m else None

    path_lower = fp.lower()
    if "cloudy" in path_lower:
        condition = "cloudy"
    elif "piper" in path_lower:
        condition = "piper"
    else:
        condition = "unknown"
    return dyad_id, condition

# -----------------------------
# 2) Clean + parse SRT text
# -----------------------------
def clean_htmlish(text: str) -> str:
    text = re.sub(r"<[^>]+>", "", str(text))
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def parse_speaker_and_utt(line: str):
    raw = str(line).replace("\n", " ").strip()

    # Pattern 1: <v Speaker>text (common in WebVTT-ish captions embedded in SRT)
    m = re.match(r"^<v\s+([^>]+)>(.*)$", raw, flags=re.IGNORECASE)
    if m:
        speaker = clean_htmlish(m.group(1)).strip()
        utt = clean_htmlish(m.group(2)).strip()
        return (speaker if speaker else None), utt

    cleaned = clean_htmlish(raw)

    # Pattern 2: [Speaker]: text
    m = re.match(r"^\[(.*?)\]\s*:\s*(.*)$", cleaned)
    if m:
        return m.group(1).strip(), m.group(2).strip()

    # Pattern 3: SPEAKER: text (all caps or Title Case; avoid matching timestamps like 00:00)
    m = re.match(r"^([A-Za-z][A-Za-z .'-]{0,40})\s*:\s*(.+)$", cleaned)
    if m:
        candidate = m.group(1).strip()
        utt = m.group(2).strip()
        if not re.match(r"^\d{1,2}$", candidate):
            return candidate, utt

    return None, cleaned

def load_one_srt(fp: str):
    dyad_id, condition = parse_filename_from_path(fp)
    subs = pysrt.open(fp)

    rows = []
    for sub in subs:
        raw = sub.text.replace("\n", " ")
        speaker, utt = parse_speaker_and_utt(raw)
        rows.append({
            "file": os.path.basename(fp),
            "path": fp,
            "dyad_id": dyad_id,
            "condition": condition,
            "start": sub.start.ordinal / 1000.0,
            "end": sub.end.ordinal / 1000.0,
            "speaker": speaker if speaker else "UNKNOWN",
            "utt": utt
        })
    df = pd.DataFrame(rows)
    df["duration"] = df["end"] - df["start"]
    return df

# -----------------------------
# 3) Collect all SRT files
# -----------------------------
all_files = []
for root, _, files in os.walk(SRT_DIR):
    for f in files:
        if f.lower().endswith(".srt"):
            all_files.append(os.path.join(root, f))

all_files = sorted(all_files)
print("Found .srt files:", len(all_files))
print("First 5:", all_files[:5])

if len(all_files) == 0:
    raise FileNotFoundError(f"No .srt files found under {SRT_DIR}")

Found .srt files: 47
First 5: ['/Users/Meihui/Downloads/sync/CVS/recording/all_srt/cloudy/dyad10_250916_cloudy_discussion.srt', '/Users/Meihui/Downloads/sync/CVS/recording/all_srt/cloudy/dyad12_250928_cloudy_discussion.srt', '/Users/Meihui/Downloads/sync/CVS/recording/all_srt/cloudy/dyad13_250928_cloudy_discussion.srt', '/Users/Meihui/Downloads/sync/CVS/recording/all_srt/cloudy/dyad14_251002_cloudy_discussion.srt', '/Users/Meihui/Downloads/sync/CVS/recording/all_srt/cloudy/dyad15_251005_cloudy_discussion.srt']


In [2]:
# -----------------------------
# 4) Write labeled TXT (one line per utterance)
# -----------------------------

def merge_adjacent_utterances(df: pd.DataFrame, max_gap_seconds: float = 0.75) -> pd.DataFrame:
    """Optional helper: merge very short gaps within the *same* speaker.
    Not used by default so that turn-taking stays visible.
    """
    df = df.sort_values(["start", "end"], kind="stable").reset_index(drop=True)
    merged = []

    for r in df.to_dict("records"):
        if not merged:
            merged.append(r)
            continue

        prev = merged[-1]
        same_speaker = (r["speaker"] == prev["speaker"])
        close_in_time = (float(r["start"]) - float(prev["end"])) <= float(max_gap_seconds)

        if same_speaker and close_in_time:
            prev["utt"] = (str(prev["utt"]) + " " + str(r["utt"])).strip()
            prev["end"] = max(float(prev["end"]), float(r["end"]))
        else:
            merged.append(r)

    return pd.DataFrame(merged)


def srt_to_labeled_txt(
    srt_path: str,
    txt_path=None,
    *,
    merge_adjacent: bool = False,
    max_gap_seconds: float = 0.75,
    drop_exact_duplicates: bool = True,
) -> str:
    df = load_one_srt(srt_path)
    if merge_adjacent:
        df = merge_adjacent_utterances(df, max_gap_seconds=max_gap_seconds)

    if drop_exact_duplicates:
        # Remove exact duplicates of (speaker, utt, start, end) to avoid repeated captions
        df = df.drop_duplicates(subset=["speaker", "utt", "start", "end"], keep="first")

    if txt_path is None:
        txt_path = re.sub(r"\.srt$", ".txt", srt_path, flags=re.IGNORECASE)

    os.makedirs(os.path.dirname(txt_path) or ".", exist_ok=True)

    with open(txt_path, "w", encoding="utf-8") as f:
        last_speaker = None
        last_utt = None
        for _, row in df.iterrows():
            speaker = str(row["speaker"]).strip() if pd.notna(row["speaker"]) else "UNKNOWN"
            utt = str(row["utt"]).strip() if pd.notna(row["utt"]) else ""
            if not utt:
                continue

            # Skip if this line is an exact repeat of the previous (same speaker and text)
            if drop_exact_duplicates and speaker == last_speaker and utt == last_utt:
                continue

            f.write(f"{speaker}: {utt}\n")
            last_speaker, last_utt = speaker, utt

    return txt_path


# Example: convert ONE file
example_srt = all_files[0]
example_txt = srt_to_labeled_txt(example_srt)
print("Wrote:", example_txt)


# Example: convert ALL files (writes next to each .srt as .txt)
written = []
for fp in all_files:
    written.append(srt_to_labeled_txt(fp))

print("Converted:", len(written), "files")
print("Last:", written[-1] if written else None)


Wrote: /Users/Meihui/Downloads/sync/CVS/recording/all_srt/cloudy/dyad10_250916_cloudy_discussion.txt
Converted: 47 files
Last: /Users/Meihui/Downloads/sync/CVS/recording/all_srt/piper/dyad9_250529_piper_discussion.txt
